In [22]:
import tiktoken
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.nn.functional as F
import urllib.request as req

In [23]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

In [24]:
try:
    with req.urlopen(url) as response:
        raw_text = response.read().decode('utf-8')
    print("Total Length", len(raw_text))
except:
    print("Text download failed")

Total Length 1115394


In [25]:
corpus = raw_text[:150]
print("Corpus Excerpt: \n", "="*30, "\n", corpus)

Corpus Excerpt: 
 First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

A


In [26]:
enc = tiktoken.encoding_for_model("gpt-4o")
token_ids = enc.encode(raw_text)
vocab_size = enc.max_token_value + 1

In [27]:
data_tensor = torch.tensor(token_ids, dtype=torch.long)

Parameters

In [ ]:
block_size = 128
batch_size = 16
top_k = 2
num_of_experts = 10
n_layers = 4
d_model = 64
d_ff = 4 * d_model
learning_rate = 1e-3
max_iters = 500
eval_interval = 100
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Train/Val Split

In [29]:
n = int(0.9 * len(data_tensor))
train_data = data_tensor[:n]
val_data = data_tensor[n:]

Batching

In [30]:
def get_batch(split, device):
    data = train_data if split == "train" else val_data
    ix = torch.randint(0, len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

Verifying Batching

In [32]:
x, y = get_batch("train", device)
assert x.shape == (batch_size, block_size)
assert y.shape == (batch_size, block_size)

Noisy TopK Router

In [ ]:
class NoisyTopKRouter(nn.Module):
    def __init__(self, d_model: int, num_of_experts: int, top_k: int = 2):
        super().__init__()
        self.num_of_experts = num_of_experts
        self.top_k = top_k
        self.gate = nn.Linear(d_model, num_of_experts, bias=False)
        self.w_noise = nn.Linear(d_model, num_of_experts, bias=False)
    
    def forward(self, x_flat: torch.Tensor):
        clean_logits = self.gate(x_flat)

        if self.training:
            noise_scale = F.softplus(self.w_noise(x_flat))
            noise = torch.rand_like(clean_logits) * noise_scale
            logits = clean_logits + noise
        else:
            logits = clean_logits
        
        probs = torch.softmax(logits, dim=-1)
        topk_weights, topk_indices = torch.topk(probs, self.top_k, dim=-1)
        topk_weights = topk_weights / topk_weights.sum(dim=-1, keepdim=True)

        importance = probs.sum(dim=0)
        top1_indices = topk_indices[:, 0]
        load = torch.bincount(top1_indices, minlength=self.num_of_experts).float()
        num_tokens = x_flat.size(0)
        aux_loss = self.num_of_experts * torch.sum((importance / num_tokens) * (load / num_tokens))
        return topk_weights, topk_indices, aux_loss

Expert

In [37]:
class Expert(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.act = nn.GELU()
        self.w2 = nn.Linear(d_ff, d_model)
    
    def forward(self, x):
        return self.w2(self.act(self.w1(x)))

MOELayer

In [38]:
class MOELayer(nn.Module):
    def __init__(self, d_model: int, d_ff: int, num_experts: int, top_k: int):
        super().__init__()
        self.num_of_experts = num_experts
        self.top_k = top_k
        self.router = NoisyTopKRouter(d_model, num_experts, top_k)
        self.experts = nn.ModuleList([Expert(d_model, d_ff) for _ in range(num_of_experts)])
    
    def forward(self, x: torch.Tensor):
        orig_shape = x.shape[-1]
        x_flat = x.view(-1, orig_shape)
        topk_weights, topk_indices, aux_loss = self.router(x_flat)
        out_flat = torch.zeros_like(x_flat)

        for i, expert in enumerate(self.experts):
            token_indices, topk_positions = (topk_indices == 1).nonzeros(as_tuple=True)
            if token_indices.numel() == 0:
                continue

            expert_outputs = expert(x_flat[token_indices])

            weights = topk_weights[token_indices, topk_positions].unsqueeze(-1)
            out_flat[token_indices] += weights * expert_outputs
        return out_flat.view(orig_shape), aux_loss